# Organ Placement Verification

The purpose of this notebook is to see if cutting CT scans with more than 128 slices to the middle 128 does not result in cutting out parts of important organs. This is done through the use of provide structure masks.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from monai.transforms import Compose, EnsureChannelFirstd, LoadImaged, Orientationd
from tqdm import tqdm

In [2]:
# Find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# Set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\couch\Documents\robust-radiotherapy-planning


In [3]:
from src.data_utils import create_data_split_dict

In [4]:
# Load data dictionary
data_dict = create_data_split_dict(
    seed=42,
    train_fraction=0.8,
    folds=5,
    save=False,
)

In [5]:
# Create the preprocessing pipeline
preprocessing = Compose(
    [
        LoadImaged(keys=["image"]),
        EnsureChannelFirstd(keys="image"),
        Orientationd(keys=["image"], axcodes="RAS", labels=None),
    ]
)

labels = [2, 3, 4, 5]

label_max_deviations = {i: [] for i in labels}

for path in tqdm([*data_dict[0]["train"], *data_dict[0]["val"]]):
    # Run the pipeline for the mask corresponding to the given CT
    path = path.replace("CT", "STRUCTURES")
    if not Path(path).exists():
        continue

    image_dict = {"image": path}
    processed_data = preprocessing(image_dict)

    # Extract numpy array
    image_np = processed_data["image"].as_tensor().numpy()

    # Calculate the middle index of the depth
    middle_depth = image_np.shape[3] // 2

    for label in labels:
        # Find all 3D coordinates where the current mask exists
        slices_with_mask = np.argwhere(image_np == label)

        if len(slices_with_mask) > 0:
            depth_indices = slices_with_mask[:, 3]

            # Calculate how many slices the mask deviates from the middle slice
            min_depth = np.min(depth_indices)
            max_depth = np.max(depth_indices)
            middle_deviation = max(max_depth - middle_depth, middle_depth - min_depth)

            label_max_deviations[label].append(middle_deviation)
        else:
            label_max_deviations[label].append(0)

label_data = pd.DataFrame(label_max_deviations)
label_data.head()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1631/1631 [08:09<00:00,  3.33it/s]


,2,3,4,5
0,55,35,45,51
1,56,36,46,52
2,55,36,45,51
3,55,35,46,51
4,55,35,46,51


In [6]:
label_data.describe()

,2,3,4,5
count,1619.000000,1619.000000,1619.000000,1619.000000
mean,42.948734,22.974058,33.969734,41.537369
std,4.243860,5.178931,4.229182,3.809143
min,34.000000,9.000000,23.000000,31.000000
25%,40.000000,19.000000,31.000000,39.000000
50%,43.000000,24.000000,34.000000,41.000000
75%,45.000000,27.000000,37.000000,44.000000
max,56.000000,36.000000,46.000000,52.000000


Since the data is cropped or padded to be exactly 128 slices (keeping the same center), all masks are guaranteed to fit, since all mask are within 56 slices of the center.